# **Безтипове лямбда-числення. _Розширення_**

Цей блокнот розширює визначення класів `Term`, `Atm`, `App` та `Abs`, додаючи наступне.

1. Властивість `var_map`, що повертає словник, ключами якого є змінні терму, а значенням для змінної-ключа є
   - 1 за умови, що ключ має в термі тількі вільні входження,
   - 2 за умови, що ключ має в термі тількі зв'язані входження,
   - 3 за умови, що ключ має в термі як вільні, так і зв'язані входження,

   маючі на увазі, що _змінна має вільне входження_ в
   - атомарний терм, якщо її ім'я співпадає з полем `_name` терму,  
   - терм-застосування, якщо вона має вільне входження в підтерм, на який посилається поле `_fun` терму або вона має вільне входження в підтерм, на який посилається поле `_val` терму,
   - терм-абстракцію, якщо її ім'я не співпадає з полем `_name` терму та вона має вільне входження в підтерм, на який посилається поле `_body` терму,

   в той час як
   - будь-яка змінна не має жодного зв'язаного входження в атомарний терм
   
   і змінна має _зв'язане входження_ в
   - терм-застосування, якщо вона має зв'язане входження в підтерм, на який посилається поле `_fun` терму або вона має зв'язане входження в підтерм, на який посилається поле `_val` терму,
   - терм-абстракцію, або якщо її ім'я співпадає з полем `_name` терму, або вона має зв'язане входження в підтерм, на який посилається поле `_body` терму.
2. Метод `rename()`, який створює новий терм за вже існуючим, шляхом перейменування кожного вільного входження змінної на іншу змінну.  
У випадку, якщо перейменування неможливе, повертається `None`.
3. Метод `substitute()`, який створює новий терм за вже існуючим, шляхом підстановки замість кожного вільного входження певної змінної іншого терму.  
У випадку, якщо підстановка неможлива, повертається `None`.


## **Імпорт та визначення модулів та фінкцій, що використовуються**

In [1]:
import typing
import re

## **Змінні (імена)**

Змінні є цеглинками для побудови лямбда-виразів (термів).

Вважаємо, що

1. можливо побудувати будь-яку скінченну множину змінних;
2. між змінними та  їх іменами є взаємно-однозначна відповідність.

Технічно це реалізується так:

3. ім'я змінної є рядком, що починається з малої латинської літери, далі він може містити будь-яку кількість латинських літер (великих і малих), десяткових цифр та символів '_';
4. кожна змінна може розміщуватися лише в одному місті пам'яті.

In [2]:
class Var(str):

    __registry: dict[str, 'Var'] = {}              # реєстр використаних імен
    __pattern = re.compile(r"[a-z][a-zA-Z0-9_]*")  # шаблон імені змінної

    @classmethod
    def checkname(cls, name: object) -> str:
        """
        Метод перевіряє, чи може об'єкт бути використаний як ім'я змінної, тобто
        1) він є рядком;
        2) рядок має необхідний формат.

        Метод викидає
            виключення TypeError, якщо не виконана умова 1) та
            виключення ValueError, якщо не виконана умова 2).

        У разі виконання обох умов метод повертає name.
        """
        if not isinstance(name, str):
            # Помилка: тип 'name' не є 'str'
            raise TypeError(
                f"Type of 'name' must be 'str' but obtained {type(name)}")
        if not cls.__pattern.fullmatch(name):
            # Помилка: 'name' не відповідати шаблону 'Var.__pattern'
            raise ValueError("Invalid 'name' format")
        return name

    def __new__(cls, name: str) -> 'Var':
        """
        Метод створює та реєструє змінну з ім'ям name, якщо такої змінної ще
        немає в реєстрі, та повертає її.
        Метод повертає змінну з ім'ям name з реєстру, якщо вона там є.

        Метод може викинути виключення при виконання cls.checkname(name).
        """
        if cls.checkname(name) not in cls.__registry:
            # Створення та реєстрація в реєстрі 'Var.__registry' змінної на ім'я
            # name, якщо така змінна ще не зареєстрована
            cls.__registry[name] = super().__new__(cls, name)
        return cls.__registry[name]

    @classmethod
    def _show_registry(cls) -> dict[str, 'Var']:
        """
        Технічний метод для контролю реєстру - повертає словник реєстру.
        """
        return cls.__registry

    @classmethod
    def fresh(cls, base: str = "x") -> 'Var':
        """
        Метод створює абсолютно 'свіжу' (таку, яка ще не використовувалася)
        змінну з ім'ям виду 'base + _ + str(i)', де i є версією - цілим не
        меншим за 1.

        Метод повертає створену змінну.
        """
        # Перевіряємо базу, якщо вона "бита" — встановлюємо її як "x"
        try:
            cls.checkname(base)
        except (TypeError, ValueError):
            base = "x"
        # Якщо ім'я base ще не використано — беремо його
        if base not in cls.__registry:
            return cls(base)
        # Інакше додаємо до нього ще неіснуючий суфікс _1, _2, ...
        i = 1
        while True:
            candidate = f"{base}_{i}"
            if candidate not in cls.__registry:
                return cls(candidate)
            i += 1

Приклади реєстрації змінної

In [3]:
for name in ['x', 1, 'y_1', '_x', 'zZ', '1x']:
    print("Змінну на ім'я", end=" ")
    try:
        _ = Var(name)
        print(f"'{name}' зареєстровано.")
    except TypeError:
        print(f"{name} відхилено через помилку типу.")
    except ValueError:
        print(f"'{name}' відхилено через помилку формату.")
print(f"Реєстр: {Var._show_registry()}")

Змінну на ім'я 'x' зареєстровано.
Змінну на ім'я 1 відхилено через помилку типу.
Змінну на ім'я 'y_1' зареєстровано.
Змінну на ім'я '_x' відхилено через помилку формату.
Змінну на ім'я 'zZ' зареєстровано.
Змінну на ім'я '1x' відхилено через помилку формату.
Реєстр: {'x': 'x', 'y_1': 'y_1', 'zZ': 'zZ'}


## **Лямбда-вирази або терми**

Лямбда-вирази (терми) є громадянами першого класу лямбда-числення.  
Синтаксично, вони конструюються у відповідності до такої граматики

```
term ::= atom | application | abstraction
atom ::= Atm(name)
application ::= App(term, term)
abstraction ::= Abs(name, term)

name ::= [a-z][a-zA-Z0-9_]*
```

### **Абстрактний клас `Term`**

Цей клас є родовим для всіх всіх видів лямбда-виразів: атомів (atom), застосувань (application) та функціональних абстракцій (abstraction).

In [4]:
class Term:
    """Абстрактний клас, від якого успадковуються усі різновиди термів."""

    __slots__ = ()

    def __str__(self) -> str:
        """перетворює терм на рядок."""
        raise NotImplementedError

    @property
    def var_map(self) -> dict[str, int]:
        raise NotImplementedError

    def rename(self, old_name: str, new_name: str) -> 'Term':
        raise NotImplementedError

    def substitute(self, name: str, term: typing.Self) -> 'Term':
        raise NotImplementedError

### **Клас атомарних виразів `Atm`**

Вирази цього виду є атомами (безструктурними константами) лямбда-числення.  
Такий вираз однозначно визначається своїм ім'ям - змінною.

In [5]:
class Atm(Term):
    """Клас представляє терми-атоми.
    Кожен екземпляр класу має єдиний атрибут
        _name: str - ім'я відповідної змінної.
    """

    __slots__ = ('_name', )

    def __init__(self, name: str):
        """Метод ініціалізує терм-атом, що відповідає змінній з ім'ям 'name'.
        """
        # Реєструємо змінну з ім'ям name, якщо такої ще немає.
        # Тут можливе виключення для некоректного name
        name = Var.checkname(name)
        self._name = name

    def __str__(self) -> str:
        return self._name

    @property
    def var_map(self) -> dict[str, int]:
        return {self._name: 1}

    def rename(self, old_name: str, new_name: str) -> 'Atm':
        old_name = Var.checkname(old_name)
        new_name = Var.checkname(new_name)
        return Atm(new_name) if old_name == self._name else self

    def substitute(self, name: str, term: typing.Self) -> 'Atm':
        name = Var.checkname(name)
        if not isinstance(term, Term):
            raise TypeError("A variable can be substituted only with a term.")
        return term if name == self._name else self


### **Клас виразів-застосувань `App`**

Вирази цього виду формалізують фразу "застосувати функцію, представлену термом $f$ до значення представленого термом $v$".

In [6]:
class App(Term):
    """Клас представляє терми-застосування.
    Кожен екземпляр класу має атрибути
        '_fun' - терм, що застосовується
        '_val' - терм, до якого застосовується
    """

    __slots__ = ('_fun', '_val')

    def __init__(self, f: Term, v: Term):
        """Метод ініціалізує терм-застосування, що визначається як результат
        застосування терму 'f' до терму 'v'.
        """
        if not (isinstance(f, Term) and isinstance(v, Term)):
            # Помилка, обидва аргументи повинні мати тип 'Term'
            raise TypeError("Both arguments must be 'Term'")
        self._fun = f
        self._val = v

    def __str__(self) -> str:
        return f"({str(self._fun)} {str(self._val)})"

    @property
    def var_map(self) -> dict[str, int]:
        temp = self._fun.var_map.copy()
        for x, status in self._val.var_map.items():
            temp[x] = 3 if x in temp and temp[x] != status else status
        return temp

    def rename(self, old_name: str, new_name: str) -> 'App':
        old_name = Var.checkname(old_name)
        new_name = Var.checkname(new_name)
        new_fun = self._fun.rename(old_name, new_name)
        new_val = self._val.rename(old_name, new_name)
        return App(new_fun, new_val)

    def substitute(self, name: str, term: typing.Self) -> typing.Self | None:
        name = Var.checkname(name)
        if not isinstance(term, Term):
            raise TypeError("A variable can be substituted only with a term.")
        new_fun = self._fun.substitute(name, term)
        new_val = self._val.substitute(name, term)
        return App(new_fun, new_val)


### **Клас функціональних абстракцій `Abs`**

Вирази цього виду моделюють функції з визначеним аргументом, який є змінною із заданеим іменем, та термом - тілом, що залежить від аргументу.

In [7]:
class Abs(Term):
    """Клас представляє терми-абстракції.
    Кожен екземпляр класу має атрибути
        '_name' - ім'я змінної - аргументу абстракції
        '_body' - терм, що представляє тіло абстракції
    """

    __slots__ = ('_name', '_body')

    def __init__(self, x: str, t: Term):
        """Метод ініціалізує терм-абстракцію, що визначається як результат
        абстрагування за
          змінною з ім'ям x та
          термом t.
        """
        if not isinstance(t, Term):
            raise TypeError("Type of 'body' must be 'Term'")
        self._body = t
        self._name = Var(x)

    def __str__(self) -> str:
        return f"(λ {self._name}. {self._body})"

    @property
    def var_map(self) -> dict[str, int]:
        temp = self._body.var_map.copy()
        if self._name in temp:
            temp[self._name] = 2
        return temp

    def rename(self, old_name: str, new_name: str) -> 'Abs':
        old_name = Var.checkname(old_name)
        new_name = Var.checkname(new_name)
        return (self if self._name in (old_name, new_name) else
                Abs(self._name, self._body.rename(old_name, new_name)))

    def substitute(self, name: str, term: typing.Self) -> 'Abs':
        name = Var.checkname(name)
        if not isinstance(term, Term):
            raise TypeError("A variable can be substituted only with a term.")
        status = self.var_map.get(name)
        if status is None or name == self._name:  # підстановка неможлива
            return self
        status = term.var_map.get(self._name)
        if status is None:  # змінна з іменем self._name не входить у term
            return Abs(self._name, self._body.substitute(name, term))
        if status & 1 == 0: # змінна з іменем name не входить вільно у term
            return Abs(self._name, self._body.substitute(name, term))
        # змінна self._name входить вільно у term
        new_name = Var.fresh(self._name)
        new_body = self._body.rename(self._name, new_name)
        new_body = self._body if new_body is None else new_body
        return Abs(new_name, new_body.substitute(name, term))


## **Приклади перейменувань**

In [8]:
t = Abs('x', App(App(Atm('x'), Atm('y')), Atm('z')))
print(f"t = {t}")
t_new = t.rename('y', 'v')
print(f"[v/y]t = {t_new}")
t_new = t.rename('x', 'u')
print(f"[u/x]t = {t_new}")
t_new = t.rename('r', 's')
print(f"[s/r]t = {t_new}")

t = (λ x. ((x y) z))
[v/y]t = (λ x. ((x v) z))
[u/x]t = (λ x. ((x y) z))
[s/r]t = (λ x. ((x y) z))


## **Приклади підстановок**

In [9]:
t = Abs('x', App(Atm('y'), Abs('u', Atm('z'))))
print(f"t = {t}")
u = Atm('u')
print(f"Терм для підстановки: {u}")
t_new = t.substitute('x', u)
print(f"[u/x]t = {t_new}")
t_new = t.substitute('y', u)
print(f"[u/y]t = {t_new}")
t_new = t.substitute('z', u)
print(f"[u/z]t = {t_new}")

t = (λ x. (y (λ u. z)))
Терм для підстановки: u
[u/x]t = (λ x. (y (λ u. z)))
[u/y]t = (λ x. (u (λ u. z)))
[u/z]t = (λ x. (y (λ u_1. u)))


## **Завдання**

### **Завдання 1**

Реалізуйте функцію `alpha_convert(t: Term, x: str, y: str) -> Term`, яка здійснює заміну _зв'язаної змінної_ `x` _свіжою змінною_ `y`.

### **Завдання 2**

Реалізуйте функцію `alpha_congruent(t1: Term, t2: Term) -> bool`, яка повертає
- `True`, якщо `t1` та `t2` є альфа-конгруєнтними, та
- `False` - у протилежному випадку.

### **Завдання 3**

Реалізуйте функцію `simplify(t: App) -> Term`, яка повертає
- результат $\beta$-редукції `t`, якщо він є редексом ($t\equiv(\lambda\ x.t_1)\,t_2$), та
- сам `t` - в іншому випадку.